<a href="https://colab.research.google.com/github/luky048-ship-it/Business-Analytics-and-AI/blob/main/Introduction%20to%20Natural%20Language%20Processing%20and%20Large%20Language%20Models/ML_learning_HW_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 — Text Preprocessing & Tokenization
**Course:** Introduction to Natural Language Processing and Large Language Models  
**Institution:** HSE University  
**Student:** Slubik Staislav Olegovich


### Disclosure of AI Assistance:
Generative AI (Gemini) was used as an educational assistant to clarify theoretical concepts, assist with boilerplate code structure, and polish the written English analysis. All code, data choices, and final interpretations have been manually verified and tested.

In [1]:
!pip -q install transformers torch pandas

In [2]:
import pandas as pd
import random
import numpy as np
import torch

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
import unicodedata

# Creating two strings that look identical to a human:
# String 'a': single precomposed character 'é' (\u00e9)
# String 'b': base character 'e' followed by combining acute accent (\u0301)
a = "café"
b = "cafe\u0301"

print(f'Human view -> String A: {a} | String B: {b}')
print("Are they equal directly in Python?", a == b)
print(f"Length A (composed): {len(a)} chars | Code points: {[hex(ord(c)) for c in a]}")
print(f"Length B (decomposed): {len(b)} chars | Code points: {[hex(ord(c)) for c in b]}")

# Applying NFC normalization
a_norm = unicodedata.normalize("NFC", a)
b_norm = unicodedata.normalize("NFC", b)

print("\n--- After NFC Normalization ---")
print("Are they equal now?", a_norm == b_norm)
print(f"Length after NFC -> A: {len(a_norm)}, B: {len(b_norm)}")

Human view -> String A: café | String B: café
Are they equal directly in Python? False
Length A (composed): 4 chars | Code points: ['0x63', '0x61', '0x66', '0xe9']
Length B (decomposed): 5 chars | Code points: ['0x63', '0x61', '0x66', '0x65', '0x301']

--- After NFC Normalization ---
Are they equal now? True
Length after NFC -> A: 4, B: 4


### Analysis of Unicode Normalization:
* **When normalization helps:**
  It is essential during data preprocessing for web scraping, user search queries, and deduplication. Different operating systems (e.g., macOS often decomposes characters into NFD, while Linux/Windows use NFC) represent accented characters differently. Without normalization, identical words would be treated as completely distinct vocabulary tokens.
* **When normalization might be inappropriate:**
  Normalization can be undesirable in specialized linguistic corpora, paleography, or low-resource languages where subtle differences in diacritics represent distinct tonal, phonetic, or semantic distinctions that should not be merged.

In [5]:
from transformers import AutoTokenizer
import pandas as pd

# Load tokenizers using AutoTokenizer
bert_tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
gpt_2_tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")


dataset = [
    # English Sentences
    ("English", "Natural language processing is transforming modern technology."),
    ("English", "Transformers handle unfamiliar words differently."),
    ("English", "The weather today is unusually cold and rainy."),
    ("English", "She didn't enjoy the movie despite the critical acclaim."),
    ("English", "Unbelievable achievements require continuous dedication."),
    # Russian Sentences
    ("Russian", "Обработка естественного языка меняет современные технологии."),
    ("Russian", "Трансформеры по-разному обрабатывают незнакомые слова."),
    ("Russian", "Сегодня на улице на редкость холодная и дождливая погода."),
    ("Russian", "Ей не понравился фильм, несмотря на восторженные отзывы."),
    ("Russian", "Невероятные достижения требуют постоянного упорства.")
]


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [6]:
records = []

for lang, sentence in dataset:
    for name, tok in [("DistilBERT (WordPiece)", bert_tokenizer), ("GPT-2 (Byte-BPE)", gpt_2_tokenizer)]:
        # Extract subword token strings
        token_pieces = tok.tokenize(sentence)

        # Calulate token-per-chatacter metric
        tokens_per_char = round(len(token_pieces) / len(sentence), 3)

        records.append({
                "Language": lang,
                "Sentence": sentence,
                "Tokenizer": name,
                "Char_Count": len(sentence),
                "Token_Count": len(token_pieces),
                "Tokens_per_Char": tokens_per_char,
                "Tokens_Sample": token_pieces[:8]  # Showing first 8 tokens for readability
        })

df_comparison = pd.DataFrame(records)

# Display styled dataframe

pd.set_option('display.max_colwidth', None)
display(df_comparison[["Language", "Tokenizer", "Char_Count", "Token_Count", "Tokens_per_Char", "Tokens_Sample"]])

,Language,Tokenizer,Char_Count,Token_Count,Tokens_per_Char,Tokens_Sample
0,English,DistilBERT (WordPiece),62,8,0.129,"[natural, language, processing, is, transforming, modern, technology, .]"
1,English,GPT-2 (Byte-BPE),62,8,0.129,"[Natural, Ġlanguage, Ġprocessing, Ġis, Ġtransforming, Ġmodern, Ġtechnology, .]"
2,English,DistilBERT (WordPiece),49,6,0.122,"[transformers, handle, unfamiliar, words, differently, .]"
3,English,GPT-2 (Byte-BPE),49,7,0.143,"[Transform, ers, Ġhandle, Ġunfamiliar, Ġwords, Ġdifferently, .]"
4,English,DistilBERT (WordPiece),46,9,0.196,"[the, weather, today, is, unusually, cold, and, rainy]"
5,English,GPT-2 (Byte-BPE),46,9,0.196,"[The, Ġweather, Ġtoday, Ġis, Ġunusually, Ġcold, Ġand, Ġrainy]"
6,English,DistilBERT (WordPiece),56,12,0.214,"[she, didn, ', t, enjoy, the, movie, despite]"
7,English,GPT-2 (Byte-BPE),56,11,0.196,"[She, Ġdidn, 't, Ġenjoy, Ġthe, Ġmovie, Ġdespite, Ġthe]"
8,English,DistilBERT (WordPiece),56,6,0.107,"[unbelievable, achievements, require, continuous, dedication, .]"
9,English,GPT-2 (Byte-BPE),56,9,0.161,"[Un, bel, iev, able, Ġachievements, Ġrequire, Ġcontinuous, Ġdedication]"


In [7]:
sample_text = "Transformers handle unfamiliar words differently."

for name, tok in [("DistilBERT (WordPiece)", bert_tokenizer), ("GPT-2 (Byte-BPE)", gpt_2_tokenizer)]:
    encoded = tok(sample_text, add_special_tockens=True)
    ids = encoded["input_ids"]
    tokens = tok.convert_ids_to_tokens(ids)
    print(f"\n{name}:")
    print(f"IDs: {ids}")
    print(f"Tokens: {tokens}")


DistilBERT (WordPiece):
IDs: [101, 19081, 5047, 16261, 2616, 11543, 1012, 102]
Tokens: ['[CLS]', 'transformers', 'handle', 'unfamiliar', 'words', 'differently', '.', '[SEP]']

GPT-2 (Byte-BPE):
IDs: [41762, 364, 5412, 22594, 2456, 10338, 13]
Tokens: ['Transform', 'ers', 'Ġhandle', 'Ġunfamiliar', 'Ġwords', 'Ġdifferently', '.']


## Detailed Comparison of Two Informative Examples

### Example 1: English Word Segmentation
For the English word "unbelievable", DistilBERT splits it into `['un', '##believ', '##able']`. The `##` prefix explicitly indicates subword continuation within a single word. Meanwhile, GPT-2 segments it as `['Un', 'bel', 'iev', 'able']` and uses the `Ġ` symbol to denote leading whitespace for subsequent words.

### Example 2: Multilingual Subword Fragmentation
In the Russian sentence "Невероятные достижения...", DistilBERT fragments words down to individual characters (e.g., `['н', '##е', '##в', '##е']`) because Cyrillic roots are virtually absent from its English-centric vocabulary. GPT-2 performs even worse: it breaks 2-byte UTF-8 Cyrillic characters into raw byte representations (`['Ð', 'Ŀ', 'Ðµ']`), tripling the token count relative to English and causing a severe "token tax".

---

## Final Analytical Interpretation

A common misconception in NLP is that token IDs can be directly compared across different models. In reality, a token ID is merely an arbitrary integer index pointing to a position in a model's specific vocabulary table. For example, token ID 101 in DistilBERT represents the special classification token `[CLS]`, whereas in GPT-2, ID 101 corresponds to a completely unrelated character or byte. Because every architecture is trained with distinct pre-tokenization rules and vocabulary sizes (roughly 30,522 for DistilBERT versus 50,257 for GPT-2), numeric token IDs have no universal meaning.

Furthermore, generating fewer tokens for a sentence does not necessarily mean a tokenizer or model is superior. Tokenization involves a fundamental engineering trade-off between sequence length and vocabulary size. If a model adopts a massive vocabulary of several million words, sentences become extremely short, but the embedding layer becomes enormous, consuming excessive GPU memory and suffering from parameter sparsity on rare words. Conversely, smaller vocabularies produce longer token sequences, which increases the quadratic computational cost of self-attention mechanisms.

Finally, our empirical table highlights the cross-lingual disparity in subword tokenizers. Models trained primarily on English data impose an unfair computational penalty on non-Latin scripts, fragmenting Russian into individual characters or raw bytes. Therefore, tokenizer efficiency is primarily a reflection of training data diversity rather than inherent model quality.